# Phase 6: Evaluation

This notebook:
- Evaluates search quality using NDCG@10, MRR@10, Recall@50
- Compares Hybrid retrieval vs Hybrid + CrossEncoder reranking
- Generates metrics for portfolio/resume

## 6.1 Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from collections import defaultdict

from src.retrieval import HybridRetriever
from src.reranker import load_reranker

print("✓ Imports successful")

✓ Imports successful


## 6.2 Load Components

In [2]:
# Load retriever
retriever = HybridRetriever(indices_dir='../data/indices')
print("✓ Retriever loaded")

Loading BM25 from ../data/indices/bm25_index.pkl...
✓ BM25 loaded: 100,000 documents
Loading FAISS from ../data/indices/faiss_index.bin...
✓ FAISS loaded: 100,000 vectors
Loading embedding model...
✓ HybridRetriever initialized
  Products: 100,000
✓ Retriever loaded


In [3]:
# Load products
df_products = pd.read_parquet('../data/processed/products.parquet')
product_lookup = df_products.set_index('product_id').to_dict('index')
print(f"✓ Loaded {len(df_products):,} products")

✓ Loaded 982,641 products


In [4]:
# Load reranker
reranker = load_reranker("balanced")
print("✓ Reranker loaded")

✓ Loaded CrossEncoder: cross-encoder/ms-marco-MiniLM-L-12-v2
✓ Reranker loaded


In [5]:
# Load test labels (ground truth)
df_labels = pd.read_parquet('../data/processed/labels_test.parquet')
print(f"✓ Loaded {len(df_labels):,} test labels")
print(f"  Unique queries: {df_labels['query'].nunique():,}")
print(f"  Label distribution:\n{df_labels['esci_label'].value_counts()}")

✓ Loaded 334,462 test labels
  Unique queries: 17,701
  Label distribution:
esci_label
Exact         233242
Substitute     66140
Irrelevant     27830
Complement      7250
Name: count, dtype: int64


## 6.3 Define Evaluation Metrics

In [6]:
# Relevance mapping: ESCI labels to scores
# E (Exact) = 3, S (Substitute) = 2, C (Complement) = 1, I (Irrelevant) = 0
RELEVANCE_MAP = {
    'Exact': 3,
    'Substitute': 2,
    'Complement': 1,
    'Irrelevant': 0
}

def get_relevance(label):
    """Convert ESCI label to relevance score."""
    return RELEVANCE_MAP.get(label, 0)

print("✓ Metrics defined")

✓ Metrics defined


In [7]:
def dcg_at_k(relevances, k):
    """Calculate DCG@k."""
    relevances = relevances[:k]
    if not relevances:
        return 0.0
    return sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))

def ndcg_at_k(relevances, k):
    """Calculate NDCG@k."""
    dcg = dcg_at_k(relevances, k)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = dcg_at_k(ideal_relevances, k)
    return dcg / idcg if idcg > 0 else 0.0

def mrr_at_k(relevances, k, threshold=2):
    """Calculate MRR@k. Threshold=2 means S or E counts as relevant."""
    for i, rel in enumerate(relevances[:k]):
        if rel >= threshold:
            return 1.0 / (i + 1)
    return 0.0

def recall_at_k(relevances, k, all_relevant_count, threshold=2):
    """Calculate Recall@k."""
    if all_relevant_count == 0:
        return 0.0
    found = sum(1 for rel in relevances[:k] if rel >= threshold)
    return found / all_relevant_count

print("✓ Metrics defined")

✓ Metrics defined


## 6.4 Build Ground Truth Lookup

In [8]:
# Build query -> {product_id: relevance} lookup
ground_truth = defaultdict(dict)

for _, row in tqdm(df_labels.iterrows(), total=len(df_labels), desc="Building ground truth"):
    query = row['query']
    product_id = row['product_id']
    label = row['esci_label']
    ground_truth[query][product_id] = get_relevance(label)

print(f"✓ Ground truth for {len(ground_truth):,} queries")

Building ground truth: 100%|██████████| 334462/334462 [00:04<00:00, 78944.92it/s]

✓ Ground truth for 17,701 queries


In [9]:
# Get queries that have at least one relevant product (E or S)
eval_queries = []
for query, products in ground_truth.items():
    relevant_count = sum(1 for rel in products.values() if rel >= 2)
    if relevant_count > 0:
        eval_queries.append(query)

print(f"✓ {len(eval_queries):,} queries with relevant products")

# Sample for faster evaluation (use all for final results)
SAMPLE_SIZE = 500  # Set to len(eval_queries) for full evaluation
np.random.seed(42)
eval_sample = np.random.choice(eval_queries, size=min(SAMPLE_SIZE, len(eval_queries)), replace=False)
print(f"✓ Using {len(eval_sample)} queries for evaluation")

✓ 17,701 queries with relevant products
✓ Using 500 queries for evaluation


## 6.5 Evaluate Hybrid Retrieval (Baseline)

In [10]:
def evaluate_retrieval(queries, retriever, ground_truth, product_lookup, 
                       reranker=None, top_k_retrieve=50, top_k_eval=10):
    """
    Evaluate retrieval (optionally with reranking).
    
    Returns dict with NDCG@10, MRR@10, Recall@50, and latencies.
    """
    ndcg_scores = []
    mrr_scores = []
    recall_scores = []
    latencies = []
    
    for query in tqdm(queries, desc="Evaluating"):
        gt = ground_truth.get(query, {})
        if not gt:
            continue
        
        # Count total relevant for recall
        total_relevant = sum(1 for rel in gt.values() if rel >= 2)
        
        # Retrieve
        start = time.time()
        results = retriever.search(query, method="hybrid", top_k=top_k_retrieve)
        
        # Add product titles for reranker
        for r in results:
            pid = r['product_id']
            if pid in product_lookup:
                r['product_title'] = product_lookup[pid].get('product_title', '')
        
        # Optionally rerank
        if reranker:
            results = reranker.rerank(query, results, top_k=top_k_retrieve)
        
        latency = (time.time() - start) * 1000
        latencies.append(latency)
        
        # Get relevances for retrieved products
        relevances = [gt.get(r['product_id'], 0) for r in results]
        
        # Calculate metrics
        ndcg_scores.append(ndcg_at_k(relevances, top_k_eval))
        mrr_scores.append(mrr_at_k(relevances, top_k_eval))
        recall_scores.append(recall_at_k(relevances, top_k_retrieve, total_relevant))
    
    return {
        'NDCG@10': np.mean(ndcg_scores),
        'MRR@10': np.mean(mrr_scores),
        'Recall@50': np.mean(recall_scores),
        'Latency_mean': np.mean(latencies),
        'Latency_p95': np.percentile(latencies, 95),
        'n_queries': len(ndcg_scores)
    }

In [11]:
# Evaluate Hybrid only (baseline)
print("Evaluating Hybrid Retrieval (baseline)...")
print("=" * 50)

baseline_results = evaluate_retrieval(
    queries=eval_sample,
    retriever=retriever,
    ground_truth=ground_truth,
    product_lookup=product_lookup,
    reranker=None  # No reranking
)

print(f"\nBaseline Results (Hybrid Only):")
print(f"  NDCG@10:  {baseline_results['NDCG@10']:.4f}")
print(f"  MRR@10:   {baseline_results['MRR@10']:.4f}")
print(f"  Recall@50: {baseline_results['Recall@50']:.4f}")
print(f"  Latency:  {baseline_results['Latency_mean']:.0f}ms (p95: {baseline_results['Latency_p95']:.0f}ms)")

Evaluating Hybrid Retrieval (baseline)...


Evaluating: 100%|██████████| 500/500 [00:48<00:00, 10.32it/s]


Baseline Results (Hybrid Only):
  NDCG@10:  0.3455
  MRR@10:   0.3388
  Recall@50: 0.0735
  Latency:  97ms (p95: 157ms)


## 6.6 Evaluate Hybrid + CrossEncoder Reranking

In [12]:
# Evaluate Hybrid + CrossEncoder
print("Evaluating Hybrid + CrossEncoder Reranking...")
print("=" * 50)

reranked_results = evaluate_retrieval(
    queries=eval_sample,
    retriever=retriever,
    ground_truth=ground_truth,
    product_lookup=product_lookup,
    reranker=reranker  # With reranking
)

print(f"\nReranked Results (Hybrid + CrossEncoder):")
print(f"  NDCG@10:  {reranked_results['NDCG@10']:.4f}")
print(f"  MRR@10:   {reranked_results['MRR@10']:.4f}")
print(f"  Recall@50: {reranked_results['Recall@50']:.4f}")
print(f"  Latency:  {reranked_results['Latency_mean']:.0f}ms (p95: {reranked_results['Latency_p95']:.0f}ms)")

Evaluating Hybrid + CrossEncoder Reranking...


Evaluating: 100%|██████████| 500/500 [01:54<00:00,  4.36it/s]


Reranked Results (Hybrid + CrossEncoder):
  NDCG@10:  0.3892
  MRR@10:   0.3765
  Recall@50: 0.0735
  Latency:  229ms (p95: 340ms)


## 6.7 Compare Results

In [13]:
# Calculate improvements
def calc_improvement(baseline, improved):
    if baseline == 0:
        return 0
    return ((improved - baseline) / baseline) * 100

ndcg_imp = calc_improvement(baseline_results['NDCG@10'], reranked_results['NDCG@10'])
mrr_imp = calc_improvement(baseline_results['MRR@10'], reranked_results['MRR@10'])
recall_imp = calc_improvement(baseline_results['Recall@50'], reranked_results['Recall@50'])

print("\n" + "=" * 60)
print("COMPARISON: Hybrid vs Hybrid + CrossEncoder")
print("=" * 60)
print(f"\n{'Metric':<15} {'Hybrid':<12} {'+ Reranker':<12} {'Improvement':<12}")
print("-" * 51)
print(f"{'NDCG@10':<15} {baseline_results['NDCG@10']:<12.4f} {reranked_results['NDCG@10']:<12.4f} {ndcg_imp:+.1f}%")
print(f"{'MRR@10':<15} {baseline_results['MRR@10']:<12.4f} {reranked_results['MRR@10']:<12.4f} {mrr_imp:+.1f}%")
print(f"{'Recall@50':<15} {baseline_results['Recall@50']:<12.4f} {reranked_results['Recall@50']:<12.4f} {recall_imp:+.1f}%")
print("-" * 51)
print(f"{'Latency (ms)':<15} {baseline_results['Latency_mean']:<12.0f} {reranked_results['Latency_mean']:<12.0f} +{reranked_results['Latency_mean'] - baseline_results['Latency_mean']:.0f}ms")


COMPARISON: Hybrid vs Hybrid + CrossEncoder

Metric          Hybrid       + Reranker   Improvement 
---------------------------------------------------
NDCG@10         0.3455       0.3892       +12.7%
MRR@10          0.3388       0.3765       +11.1%
Recall@50       0.0735       0.0735       +0.0%
---------------------------------------------------
Latency (ms)    97           229          +132ms


## 6.8 Save Results

In [14]:
# Create results dataframe
results_df = pd.DataFrame([
    {
        'Configuration': 'Hybrid (BM25 + FAISS)',
        'NDCG@10': baseline_results['NDCG@10'],
        'MRR@10': baseline_results['MRR@10'],
        'Recall@50': baseline_results['Recall@50'],
        'Latency_ms': baseline_results['Latency_mean']
    },
    {
        'Configuration': 'Hybrid + CrossEncoder',
        'NDCG@10': reranked_results['NDCG@10'],
        'MRR@10': reranked_results['MRR@10'],
        'Recall@50': reranked_results['Recall@50'],
        'Latency_ms': reranked_results['Latency_mean']
    }
])

# Save to CSV
results_df.to_csv('../results/evaluation_results.csv', index=False)
print("✓ Results saved to ../results/evaluation_results.csv")
print("\n")
print(results_df.to_string(index=False))

✓ Results saved to ../results/evaluation_results.csv


        Configuration  NDCG@10   MRR@10  Recall@50  Latency_ms
Hybrid (BM25 + FAISS) 0.345465 0.338782   0.073465   96.629162
Hybrid + CrossEncoder 0.389174 0.376479   0.073465  228.769715


## 6.9 Portfolio Summary

In [15]:
print("\n" + "=" * 60)
print("📊 PORTFOLIO METRICS")
print("=" * 60)
print("\nFor your resume/portfolio, you can say:\n")
print(f"• Improved NDCG@10 by {ndcg_imp:+.1f}% using CrossEncoder reranking")
print(f"• Improved MRR@10 by {mrr_imp:+.1f}% (faster time-to-first-relevant-result)")
print(f"• Achieved {reranked_results['Recall@50']*100:.1f}% Recall@50")
print(f"• End-to-end latency: {reranked_results['Latency_mean']:.0f}ms (p95: {reranked_results['Latency_p95']:.0f}ms)")
print(f"• Evaluated on {baseline_results['n_queries']} queries from Amazon ESCI dataset")
print("\n" + "=" * 60)


📊 PORTFOLIO METRICS

For your resume/portfolio, you can say:

• Improved NDCG@10 by +12.7% using CrossEncoder reranking
• Improved MRR@10 by +11.1% (faster time-to-first-relevant-result)
• Achieved 7.3% Recall@50
• End-to-end latency: 229ms (p95: 340ms)
• Evaluated on 500 queries from Amazon ESCI dataset



## 6.10 Summary

In [ ]:
print("\n✅ Phase 6 Complete!")
print("=" * 50)
print("\nWhat we measured:")
print("  • NDCG@10  - Ranking quality")
print("  • MRR@10   - Time to first relevant result")
print("  • Recall@50 - Coverage of relevant products")
print("\nKey findings:")
print(f"  • CrossEncoder reranking improves NDCG@10 by {ndcg_imp:+.1f}%")
print(f"  • Latency cost: +{reranked_results['Latency_mean'] - baseline_results['Latency_mean']:.0f}ms")
print("\n📁 Results saved to: ../results/evaluation_results.csv")



✅ Phase 6 Complete!

What we measured:
  • NDCG@10  - Ranking quality
  • MRR@10   - Time to first relevant result
  • Recall@50 - Coverage of relevant products

Key findings:
  • CrossEncoder reranking improves NDCG@10 by +12.7%
  • Latency cost: +132ms

📁 Results saved to: ../results/evaluation_results.csv

🚀 Next: Phase 7 - Streamlit Demo
